In [1]:
from pathlib import Path
import os
import google.colab
assert Path('/content').is_dir() and os.environ.get('COLAB_RELEASE_TAG'), 'Select a Google Colab runtime'
assert not Path('/dev/nvidia0').exists() and not os.environ.get('COLAB_TPU_ADDR'), 'Select CPU / accelerator None'
%pip install -q numpy==2.2.6 scipy==1.15.3


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 35.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 16.7 MB/s eta 0:00:0000:0100:01


## 2. Restore one input ZIP from Google Drive
Upload `PP1_V3_Colab_Inputs.zip` to My Drive, then edit ZIP_PATH if needed. Keep the original ZIP; this cell only reads it. Inputs restore to `/content/PP1_workspace/`.

The cell validates every member before extraction, rejects unsafe paths and links, checks CRCs and preparation hashes, and refuses an existing destination. If interrupted, preserve the incomplete folder and use a fresh Colab runtime; do not overwrite it. Drive backups use DRIVE_BACKUP_ROOT, separately from the extracted ROOT.


In [2]:
from pathlib import Path
from google.colab import drive
import hashlib, json, os, shutil, stat, subprocess, sys, uuid, zipfile
from datetime import datetime, timezone

drive.mount('/content/drive')
ZIP_PATH = Path('/content/drive/MyDrive/PP1_V3_Colab_Inputs.zip')  # EDIT IF NEEDED
DRIVE_BACKUP_ROOT = Path('/content/drive/MyDrive/PP1_V3_runs')
ROOT = Path('/content/PP1_workspace')
if ROOT.exists() or ROOT.is_symlink():
    raise FileExistsError('Restore destination already exists; use a fresh runtime. No files overwritten.')

ENGINE_REL = 'motor_scoring_research/reference_engine/pp1_motor_engine.py'
V3_REL = 'motor_scoring_research/v3/'
ENGINE_HASH = 'c39620fe8738ca9217b1b7f9d75141e0f2ab9a7b49d22ebd63b6234202c437e4'
V2_GENERATOR_HASH = 'adef793184258e971659c001e389556786d79ff74955dd4f6cf273a754184d69'

def safe_member(name):
    parts = name.split('/')
    if (not name or name.startswith('/') or '\\' in name or ':' in name
            or any(part in ('', '.', '..') for part in parts)
            or any(ord(ch) < 32 for ch in name)):
        raise ValueError(f'Unsafe ZIP path: {name!r}')
    if not (name.startswith('PADS + MJFF/') or name == ENGINE_REL or name.startswith(V3_REL)):
        raise ValueError(f'Unexpected ZIP member: {name}')
    target = ROOT.joinpath(*parts)
    if not target.resolve().is_relative_to(ROOT.resolve()):
        raise ValueError(f'ZIP path escapes destination: {name}')
    return target

with zipfile.ZipFile(ZIP_PATH) as archive:
    members = archive.infolist()
    if len(members) > 10000 or sum(m.file_size for m in members) > 2_000_000_000:
        raise ValueError('Bundle exceeds expected input limits')
    names = set()
    targets = []
    for member in members:
        target = safe_member(member.filename)
        mode = member.external_attr >> 16
        if member.is_dir() or stat.S_IFMT(mode) not in (0, stat.S_IFREG):
            raise ValueError('Only regular files are accepted; directories, links and devices are refused')
        if member.flag_bits & 1:
            raise ValueError('Encrypted ZIP members are not supported')
        normalized = member.filename.casefold()
        if normalized in names:
            raise ValueError('Duplicate/case-colliding ZIP member')
        names.add(normalized)
        targets.append((member, target))
    for name in names:
        parts = name.split('/')
        if any('/'.join(parts[:i]) in names for i in range(1, len(parts))):
            raise ValueError('File/directory path conflict')
    required = {ENGINE_REL, V3_REL+'pp1_v3.py', V3_REL+'verify_v2.py',
                V3_REL+'manifests/preparation_lock.json', V3_REL+'configs/simulation_v3.json',
                V3_REL+'notebooks/Generate_and_Audit_V3.ipynb',
                'PADS + MJFF/dataset_manifest.json', 'PADS + MJFF/regenerate_dataset.py',
                'PADS + MJFF/features_extracted_by_existing_PP1_engine.csv',
                'PADS + MJFF/demo_rule_scores_for_fusion.csv',
                'PADS + MJFF/sessions_and_provenance.csv'}
    if not required.issubset(set(archive.namelist())):
        raise ValueError('Required input files are missing')
    bad = archive.testzip()
    if bad is not None:
        raise ValueError(f'ZIP CRC failure: {bad}')
    if hashlib.sha256(archive.read(ENGINE_REL)).hexdigest() != ENGINE_HASH:
        raise ValueError('Original engine hash mismatch')
    if hashlib.sha256(archive.read('PADS + MJFF/regenerate_dataset.py')).hexdigest() != V2_GENERATOR_HASH:
        raise ValueError('Original V2 generator hash mismatch')
    preparation = json.loads(archive.read(V3_REL+'manifests/preparation_lock.json'))
    for relative, expected_hash in preparation['files'].items():
        member_name = V3_REL + relative
        safe_member(member_name)
        if hashlib.sha256(archive.read(member_name)).hexdigest() != expected_hash:
            raise ValueError(f'Preparation hash mismatch: {relative}')
    ROOT.mkdir(exist_ok=False)
    for member, target in targets:
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, target.open('xb') as destination:
            shutil.copyfileobj(source, destination, length=1024*1024)

V3 = ROOT / 'motor_scoring_research' / 'v3'
ENGINE = ROOT / ENGINE_REL
V2 = ROOT / 'PADS + MJFF'
sys.dont_write_bytecode = True
sys.path.insert(0, str(V3))
import pp1_v3 as v3
v3.require_colab_cpu()
v3.verify_preparation(V3 / 'configs/simulation_v3.json')
config = v3.read_config(V3 / 'configs/simulation_v3.json')
assert v3.sha256(ENGINE) == config['engine_sha256'], 'Engine changed: stop'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
OUT = Path('/content') / (config['dataset_name'] + '_' + RUN_ID)
V2_REPORT = Path('/content') / ('V2_reproduction_' + RUN_ID + '.json')
print('Restored and verified:', ROOT)
print('Planned output:', OUT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Restored and verified: /content/PP1_workspace
Planned output: /content/PP1_Synthetic_Motor_Dataset_V3_20261020T210904Z_b8c13141


## 3. V2 reproduction gate ? approval required
Full raw feature extraction and scoring occurs here, only in Colab. Leave FALSE until explicitly approved. The report distinguishes six-decimal CSV rounding from changed features or labels. A mismatch must be investigated; do not edit scores or thresholds to pass.


In [3]:
APPROVE_V2_VERIFICATION = True
if APPROVE_V2_VERIFICATION:
    subprocess.run([sys.executable, str(V3/'verify_v2.py'), '--v2', str(V2),
                    '--engine', str(ENGINE), '--report', str(V2_REPORT)], check=True)
    print(json.dumps(json.loads(V2_REPORT.read_text()), indent=2))
else:
    print('V2 full verification disabled. No raw processing performed.')


{
  "sessions": 200,
  "engine_sha256": "c39620fe8738ca9217b1b7f9d75141e0f2ab9a7b49d22ebd63b6234202c437e4",
  "feature_absolute_tolerance": 2e-06,
  "max_feature_absolute_error": {
    "ts_tremor_rms_ms2": 3.0343948548283706e-08,
    "bs_tap_count": 0.0,
    "bs_tap_rate_per_min": 0.0,
    "bs_tap_interval_std_s": 4.991798752997101e-09,
    "rs_mean_gyro_peak_rad_s": 2.749559019354919e-07,
    "rs_movement_peaks": 0.0,
    "gi_step_candidates": 0.0,
    "gi_cadence_steps_min": 3.3333350302200415e-08,
    "gi_step_interval_std_s": 4.9797957138095206e-09
  },
  "feature_mismatches": [],
  "label_mismatches": [],
  "passed": true,
  "interpretation": "Raw-to-stored reproduction, not clinical or ML accuracy"
}


## 4. Review frozen design
240 main sessions: 168 train / 36 validation / 36 test. All four recordings remain together. Profiles are simulated, never diagnoses. No empirical anchors are reused and no source independence is claimed. Review the configuration before approving; altering the frozen preparation requires a separately reviewed version, not silent test-driven tuning.


In [4]:
print(json.dumps(config, indent=2))
print((V3/'configs/evaluation_protocol_v3.json').read_text())
print('No generation has been requested by this cell.')


{
  "dataset_name": "PP1_Synthetic_Motor_Dataset_V3",
  "generator_version": "3.0.0",
  "seed": 20261021,
  "sessions_per_profile": 120,
  "split_counts_per_profile": {
    "train": 84,
    "validation": 18,
    "test": 18
  },
  "engine_sha256": "c39620fe8738ca9217b1b7f9d75141e0f2ab9a7b49d22ebd63b6234202c437e4",
  "sampling_hz": 100,
  "durations_s": {
    "TS": 10,
    "BS": 10,
    "RS": 10,
    "GI": 180
  },
  "source_policy": "assumptions_only_no_empirical_numeric_anchors",
  "independence_claim": "within_generator_only",
  "families": [
    "continuous",
    "amplitude_modulated",
    "intermittent"
  ],
  "healthy_beta": [
    1.2,
    4.0
  ],
  "symptom_beta": [
    1.8,
    1.5
  ],
  "shared_latent_weight": 0.25,
  "domain_latent_weight": 0.55,
  "broad_uniform_weight": 0.2,
  "ts_amplitude_ms2": [
    0.035,
    2.6
  ],
  "ts_frequency_hz": [
    4.0,
    9.0
  ],
  "ts_gravity_axis_cosine": [
    0.4,
    1.0
  ],
  "bs_rate_hz": [
    0.7,
    4.5
  ],
  "bs_interval_cv

## 5. Generate and verify V3 ? explicit approval required
This creates one new dataset and then audits its saved CSVs against the unchanged engine. It does not train a model. It never retries to obtain favorable labels, class coverage, or accuracy. A failed run is incomplete: preserve it and use a new run ID only after investigating the failure.


In [5]:
APPROVE_FULL_GENERATION = True
if APPROVE_FULL_GENERATION:
    assert APPROVE_V2_VERIFICATION and V2_REPORT.is_file(), 'First complete the V2 audit'
    assert json.loads(V2_REPORT.read_text())['passed'], 'Resolve V2 mismatches first'
    subprocess.run([sys.executable, str(V3/'pp1_v3.py'), 'generate',
                    '--config', str(V3/'configs/simulation_v3.json'), '--engine', str(ENGINE),
                    '--output', str(OUT), '--approve-generation'], check=True)
else:
    print('Full V3 generation disabled. Await explicit approval.')


## 6. Inspect integrity and observed class coverage
These are integrity/coverage counts, not ML performance. Missing or rare classes are limitations, not a reason to silently regenerate. No test predictions are used for model selection. COMPLETE.json appears only if the audit passes.


In [6]:
if (OUT/'COMPLETE.json').is_file():
    print((OUT/'integrity_report.json').read_text())
else:
    print('No complete V3 dataset exists for this run.')


{
  "status": "passed",
  "session_count": 240,
  "complete_score_sessions": 240,
  "complete_score_coverage": 1.0,
  "class_counts_0_to_4": {
    "test": {
      "TS": [
        18,
        10,
        3,
        2,
        3
      ],
      "BS": [
        4,
        9,
        15,
        6,
        2
      ],
      "RS": [
        18,
        6,
        7,
        4,
        1
      ],
      "GI": [
        16,
        6,
        3,
        8,
        3
      ]
    },
    "train": {
      "TS": [
        100,
        24,
        20,
        16,
        8
      ],
      "BS": [
        39,
        55,
        33,
        24,
        17
      ],
      "RS": [
        78,
        33,
        22,
        23,
        12
      ],
      "GI": [
        61,
        42,
        27,
        18,
        20
      ]
    },
    "validation": {
      "TS": [
        24,
        3,
        5,
        2,
        2
      ],
      "BS": [
        7,
        13,
        9,
        3,
        4
      ],

## 7. Optional predefined generator-shift set ? disabled
40 separate sessions, same simulator foundations with chirp/noise/interference shifts. No tuning on its results. This is not an external clinical test set. Its configuration was fixed before generation.


In [7]:
APPROVE_STRESS_GENERATION = False
STRESS_OUT = Path(str(OUT) + '_stress')
if APPROVE_STRESS_GENERATION:
    assert APPROVE_FULL_GENERATION and (OUT/'COMPLETE.json').is_file()
    subprocess.run([sys.executable, str(V3/'pp1_v3.py'), 'generate', '--stress',
                    '--config', str(V3/'configs/simulation_v3.json'), '--engine', str(ENGINE),
                    '--output', str(STRESS_OUT), '--approve-generation'], check=True)
else:
    print('Optional stress generation disabled.')


Optional stress generation disabled.


## 8. Back up completed outputs to a fresh Drive directory
This copies files; no prior runs are overwritten. Includes the configuration, code/engine snapshots, source manifests, scores and audit. Compare copied file hashes before considering the backup complete.


In [8]:
import shutil
if (OUT/'COMPLETE.json').is_file():
    BACKUP = DRIVE_BACKUP_ROOT / OUT.name
    assert not BACKUP.exists(), 'Backup already exists: refuse overwrite'
    BACKUP.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(OUT, BACKUP)
    for original in OUT.rglob('*'):
        if original.is_file():
            assert v3.sha256(original) == v3.sha256(BACKUP/original.relative_to(OUT))
    if V2_REPORT.is_file():
        shutil.copy2(V2_REPORT, BACKUP/V2_REPORT.name)
    if (STRESS_OUT/'COMPLETE.json').is_file():
        stress_backup = DRIVE_BACKUP_ROOT / STRESS_OUT.name
        assert not stress_backup.exists()
        shutil.copytree(STRESS_OUT, stress_backup)
        for original in STRESS_OUT.rglob('*'):
            if original.is_file():
                assert v3.sha256(original) == v3.sha256(stress_backup/original.relative_to(STRESS_OUT))
    print('Backup verified:', BACKUP)
else:
    print('No completed dataset to back up.')


Backup verified: /content/drive/MyDrive/PP1_V3_runs/PP1_Synthetic_Motor_Dataset_V3_20261020T210904Z_b8c13141


## Stop ? no model training
Future Phase 2B will compare RF, XGBoost, scaled RBF SVM and CatBoost separately for each domain (16 candidates). All preprocessing/search is training-only; validation selects under the frozen protocol; test is evaluated once after model/configuration freeze. The rule-engine baseline defines demo labels. Any accuracy is synthetic demo-label agreement, never clinical accuracy. Real-device mode remains features-only.
